[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AndresOrdonez369/deepLearningWeek1/blob/main/proyectos/NB01_viernes_tu_primera_neurona.ipynb)

# NB01 · Tu primera neurona
**Neural Networks: Deep Learning · UIDE · Clase 1 · Viernes** — Docente: Andrés Ordóñez-Bolaños

En este notebook vas a **construir una neurona con tus propias manos**, verla tomar decisiones, equivocarse, aprender de sus errores y bajar una "montaña" con descenso de gradiente. No necesitas instalar nada.

### Cómo usar este notebook (2 minutos)
0. Si lo abriste desde GitHub: **Archivo → Guardar una copia en Drive**. Así tus cambios quedan guardados.
1. Ejecuta la celda **Configuración** (la primera de código). Luego sigue en orden con `Shift + Enter`.
2. Cada sección tiene dos partes:
   - **Ejemplo resuelto** — ya está hecho: léelo, ejecútalo y compara con **Qué deberías ver**.
   - **Tarea** — te toca a ti: reemplaza los `None` y ejecuta la celda **Revisar**, que te dice ✅ o ❌ con una pista.
3. Al final está el **Mini-reto 1**: diseña tu propio comité de crédito.
4. Las tareas son **formativas** (no tienen nota): sirven para que compruebes que entendiste.

| Sección | Ejemplo resuelto | Tu tarea | Lámina |
|---|---|---|---|
| 1 | Una neurona como calculadora | T1 · Programa tu neurona | 18–20 |
| 2 | La activación: el interruptor | T2 · De puntaje a decisión | 22–23 |
| 3 | El perceptrón aprende de un error | T3 · Corrige un error al revés | 27–28 |
| 4 | Entrenar un perceptrón completo | T4 · Aprende la compuerta AND | 29 |
| 5 | El límite: XOR | T5 · ¿Por qué falla? | 30 |
| 6 | La pérdida | T6 · Programa MSE y BCE | 34–35 |
| 7 | Descenso de gradiente | T7 · Baja otra montaña | 36–38 |
| 8 | El tamaño del paso η | T8 · ¿Cuáles convergen? | 39 |
| 🏁 | **Mini-reto 1** | Tu propio comité de crédito | 40 |

In [ ]:
# @title Configuración y autocorrección · ejecuta esta celda primero (no necesitas abrirla)
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(2026)  # fixed seed -> everyone gets the same results
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

# ---------- self-check helpers (formative feedback, no grades) ----------
def _ok(msg):   print("✅ " + msg)
def _no(msg):   print("❌ " + msg)
def _todo():    print("⏳ Aún no completas esta tarea: reemplaza los None y vuelve a ejecutar.")
def _num(v):
    try:
        return float(np.asarray(v, dtype=float).reshape(-1)[0]) if np.size(v) == 1 else np.asarray(v, dtype=float)
    except Exception:
        return None
def _near(a, b, tol=1e-2):
    try:
        return bool(np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float), atol=tol))
    except Exception:
        return False
_sig = lambda z: 1 / (1 + np.exp(-z))

def revisar_t1(neurona, z_b):
    if neurona is None or z_b is None:
        return _todo()
    try:
        out = neurona(np.array([0.8, 0.5, -1.0]), np.array([1.2, 0.7, -2.0]), -0.5)
    except Exception as e:
        return _no(f"Tu función produjo un error: {e}")
    if out is None:
        return _no("Tu función devuelve None. Pista: return (w * x).sum() + b   o   w @ x + b")
    if not _near(out, 2.81):
        return _no(f"Con el socio del ejemplo debería dar 2.81 y da {_num(out)}. ¿Sumaste el sesgo b?")
    rng = np.random.default_rng(1)
    for _ in range(5):
        x, w, b = rng.normal(size=4), rng.normal(size=4), rng.normal()
        if not _near(neurona(x, w, b), w @ x + b, 1e-6):
            return _no("Funciona con 3 datos, pero no con otros vectores. ¿Escribiste los números a mano?")
    if not _near(z_b, -3.61):
        return _no(f"La función está bien ✔, pero z_b debería ser -3.61 y es {_num(z_b)}.")
    _ok("¡Perfecto! Tu neurona funciona y el socio B obtiene z = -3.61: el comité lo rechaza.")

def revisar_t2(p_b, decision_b):
    if p_b is None or decision_b is None:
        return _todo()
    if not _near(p_b, _sig(-3.61), 1e-3):
        return _no(f"p_b debería ser sigmoid(z_b) ≈ 0.026 y es {_num(p_b)}.")
    if str(decision_b).strip().lower() not in ("rechaza", "rechazar", "no"):
        return _no("Con 2.6 % de probabilidad de pago y umbral 0.5, la decisión es 'rechaza'.")
    _ok("Correcto: 2.6 % de probabilidad de pago → el comité rechaza.")

def revisar_t3(w_nuevo, b_nuevo, todavia_se_equivoca):
    if w_nuevo is None or b_nuevo is None or todavia_se_equivoca is None:
        return _todo()
    if not _near(w_nuevo, [0.2, 0.0]):
        return _no(f"w_nuevo debería ser w + η·y·x = (0.2, 0.0) y es {_num(w_nuevo)}. Ojo: aquí y = -1.")
    if not _near(b_nuevo, 0.0):
        return _no(f"b_nuevo debería ser b + η·y = 0.0 y es {_num(b_nuevo)}.")
    if todavia_se_equivoca is not True:
        return _no("Calcula z con los pesos nuevos: 0.2·1 + 0·0.5 + 0 = 0.2 ≥ 0, así que todavía aprueba (se equivoca).")
    _ok("¡Exacto! Una corrección acercó la frontera, pero no bastó: el perceptrón necesita varias vueltas.")

def revisar_t4(w_and, b_and, X_and, y_and):
    if w_and is None or b_and is None or X_and is None or y_and is None:
        return _todo()
    X_and, y_and = np.asarray(X_and, float), np.asarray(y_and)
    if X_and.shape != (4, 2) or sorted(map(tuple, X_and)) != [(0, 0), (0, 1), (1, 0), (1, 1)]:
        return _no("X_and debe tener las 4 combinaciones de dos interruptores: (0,0), (0,1), (1,0), (1,1).")
    truth = np.array([1 if (a == 1 and b == 1) else -1 for a, b in X_and])
    if not np.array_equal(y_and, truth):
        return _no("y_and: AND solo es +1 cuando ambos valen 1; en los otros tres casos es -1.")
    pred = np.where(X_and @ np.asarray(w_and, float) + b_and >= 0, 1, -1)
    if not np.array_equal(pred, y_and):
        return _no("Los pesos aprendidos todavía no clasifican bien los 4 casos. ¿Entrenaste con train_perceptron?")
    _ok(f"¡Aprendió AND! w = {np.round(w_and, 2)}, b = {b_and}. AND sí se puede separar con una línea.")

def revisar_t5(respuesta_t5):
    if respuesta_t5 is None:
        return _todo()
    if str(respuesta_t5).strip().lower() == "b":
        return _ok("Correcto: no existe UNA línea recta que deje (0,1) y (1,0) de un lado y (0,0) y (1,1) del otro.")
    hints = {"a": "Más épocas no ayudan: en la gráfica los errores se repiten igual para siempre.",
             "c": "Con cualquier η pasa lo mismo: el problema no es el tamaño del paso.",
             "d": "Los 4 datos de XOR no tienen ruido: son exactos."}
    _no(hints.get(str(respuesta_t5).strip().lower(), "Elige una letra: 'a', 'b', 'c' o 'd'."))

def revisar_t6(mse, bce):
    if mse is None or bce is None:
        return _todo()
    try:
        m = mse(np.array([120, 95, 150, 80]), np.array([110, 100, 170, 82]))
        b1, b0 = bce(1, 0.9), bce(0, 0.9)
    except Exception as e:
        return _no(f"Tus funciones produjeron un error: {e}")
    if m is None and b1 is None:
        return _todo()
    if m is None or b1 is None or b0 is None:
        return _no("Una de las dos funciones todavía devuelve None: completa su return.")
    if not _near(m, 132.25):
        return _no(f"mse con las casas del ejemplo debería dar 132.25 y da {_num(m)}. ¿Elevaste al cuadrado antes de promediar?")
    if not _near(b1, -np.log(0.9), 1e-3):
        return _no(f"bce(1, 0.9) debería ser -log(0.9) ≈ 0.105 y da {_num(b1)}.")
    if not _near(b0, -np.log(0.1), 1e-3):
        return _no(f"bce(0, 0.9) debería ser -log(1 - 0.9) ≈ 2.303 y da {_num(b0)}. Usa la fórmula completa con y y (1 - y).")
    _ok("¡Bien! MSE = 132.25; BCE: decir 90 % cuando sí pagó cuesta 0.11, y cuando no pagó cuesta 2.30.")

def revisar_t7(minimo, w_final):
    if minimo is None or w_final is None:
        return _todo()
    if not _near(minimo, -2):
        return _no("(w + 2)² vale 0 cuando w + 2 = 0. ¿En qué w está el fondo?")
    if not _near(w_final, -2, 0.01):
        return _no(f"Después de 50 pasos w debería estar muy cerca de -2 y está en {_num(w_final)}. Revisa la pendiente: 2·(w + 2).")
    _ok(f"¡Llegaste al fondo! w = {float(w_final):.4f} ≈ -2.")

def revisar_t8(converge):
    if converge is None or any(v is None for v in converge.values()):
        return _todo()
    truth = {0.05: True, 0.5: True, 0.9: True, 1.1: False}
    wrong = [eta for eta in truth if converge.get(eta) != truth[eta]]
    if wrong:
        return _no(f"Revisa η = {wrong}. Pista: usa la función descenso(eta) y mira si |w - 3| se achica o crece.")
    _ok("Correcto: con η < 1 se acerca (0.05 lento, 0.9 rebotando); con η = 1.1 se aleja cada vez más.")

# Mini-reto 1 data: six members (scaled income, scaled previous delays); +1 = paid, -1 = did not pay
SOCIOS = np.array([[0.9, -0.8], [0.4, -0.2], [0.8, 0.3], [-0.5, 0.6], [0.1, 0.5], [-0.3, -0.1]])
PAGO = np.array([1, 1, 1, -1, -1, -1])
NOMBRES = ["Ana", "Byron", "Carla", "Diego", "Elena", "Fabián"]

def dibujar_comite(w, b):
    w = np.asarray(w, float)
    plt.figure(figsize=(5.5, 5))
    for (x1, x2), y, n in zip(SOCIOS, PAGO, NOMBRES):
        plt.scatter(x1, x2, s=160, c="#1F55E0" if y == 1 else "#D97706", edgecolor="k", zorder=3)
        plt.annotate(n, (x1, x2), textcoords="offset points", xytext=(8, 8))
    xs = np.linspace(-1, 1, 50)
    if abs(w[1]) > 1e-9:
        plt.plot(xs, -(w[0] * xs + b) / w[1], "k-", lw=2.5, label="tu frontera (z = 0)")
    plt.xlim(-1, 1); plt.ylim(-1, 1); plt.xlabel("ingreso (escalado)"); plt.ylabel("atrasos previos (escalado)")
    plt.title("Azul = pagó · Naranja = no pagó"); plt.legend(loc="lower left"); plt.show()

def revisar_mini_reto_1(w, b):
    if w is None or b is None:
        return _todo()
    pred = np.where(SOCIOS @ np.asarray(w, float) + b >= 0, 1, -1)
    hits = int((pred == PAGO).sum())
    dibujar_comite(w, b)
    for n, p, y in zip(NOMBRES, pred, PAGO):
        print(f"  {n:7s} → {'aprueba' if p == 1 else 'rechaza'}  {'✔' if p == y else '✘ (se equivocó)'}")
    if hits == 6:
        _ok("¡6 de 6! Tu comité decide bien con estos socios. ¿Qué pesos eligieron otros grupos?")
    else:
        _no(f"{hits} de 6 aciertos. Gira la línea con los pesos y muévela con el sesgo.")

print("Listo ✔  NumPy", np.__version__, "· autocorrección cargada")

---
## 1 · Una neurona es una calculadora
**Idea:** una neurona multiplica cada dato por su **peso** (cuánto le importa), suma todo y agrega un **sesgo** (qué tan exigente es).

$$z = w_1x_1 + w_2x_2 + w_3x_3 + b \qquad\text{o, en corto,}\qquad z = \mathbf{w}\cdot\mathbf{x} + b$$

### Ejemplo resuelto: el comité de crédito
Una cooperativa evalúa a un socio con tres datos (ya escalados para que sean comparables):

| Dato | Valor $x$ | Peso $w$ | Significado del peso |
|---|---|---|---|
| Ingreso | 0.8 | 1.2 | a favor |
| Antigüedad como socio | 0.5 | 0.7 | a favor |
| Atrasos previos | −1.0 (menos que el promedio) | −2.0 | los atrasos restan |

El sesgo $b = -0.5$ representa a un comité algo exigente.

In [ ]:
x = np.array([0.8, 0.5, -1.0])    # member data: income, seniority, previous delays (scaled)
w = np.array([1.2, 0.7, -2.0])    # weights: how much the committee trusts each signal
b = -0.5                          # bias: general disposition of the committee

contributions = w * x             # each input times its weight
z = contributions.sum() + b       # weighted sum + bias

for name, c in zip(["Ingreso", "Antigüedad", "Atrasos"], contributions):
    print(f"{name:11s} aporta {c:+.2f}")
print(f"Sesgo       aporta {b:+.2f}")
print(f"Puntaje total z = {z:.2f}")

**Qué deberías ver:** aportes +0.96, +0.35 y +2.00; sesgo −0.50; **z = 2.81**.
Fíjate: un peso negativo por un valor negativo da un aporte **positivo** (tener menos atrasos que el promedio ayuda).

### Tarea 1 · Programa tu neurona
1. Completa la función `neurona(x, w, b)` para que devuelva $z = \mathbf{w}\cdot\mathbf{x} + b$ con **cualquier** vector.
2. Úsala con el **socio B**: ingreso 0.2, antigüedad −0.5, atrasos 1.5 (más atrasos que el promedio), con los mismos pesos y sesgo. Guarda el resultado en `z_b`.

<details><summary>Pista</summary>

En NumPy, `w @ x` multiplica parejas y suma (producto punto). También sirve `(w * x).sum()`.
</details>

In [ ]:
def neurona(x, w, b):
    return None          # TODO: weighted sum plus bias

x_b = np.array([0.2, -0.5, 1.5])
z_b = None               # TODO: use your function with x_b, w and b
print("z del socio B:", z_b)

In [ ]:
revisar_t1(neurona, z_b)

---
## 2 · La activación: el interruptor de la neurona
**Idea:** el puntaje $z$ todavía no es una decisión. La **función de activación** lo convierte en la salida de la neurona.

- **Escalón:** sí o no (aprueba = 1, rechaza = 0). Como un interruptor de luz.
- **Sigmoide** $\sigma(z) = \dfrac{1}{1+e^{-z}}$: convierte cualquier $z$ en un número entre 0 y 1 → se lee como **probabilidad**. Como un regulador de intensidad.
- **ReLU** $=\max(0, z)$: deja pasar lo positivo y apaga lo negativo. Es la más usada dentro de las redes. Como una válvula.

### Ejemplo resuelto

In [ ]:
step    = lambda z: np.where(z >= 0, 1.0, 0.0)
sigmoid = lambda z: 1 / (1 + np.exp(-z))
relu    = lambda z: np.maximum(0, z)

print(f"Para z = {z:.2f}:  escalón = {step(z):.0f} | sigmoide = {sigmoid(z):.3f} | ReLU = {relu(z):.2f}")

zz = np.linspace(-6, 6, 300)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for a, f, name, color in zip(ax, [step, sigmoid, relu], ["Escalón", "Sigmoide", "ReLU"], ["#1F55E0", "#B45309", "#0F766E"]):
    a.plot(zz, f(zz), lw=3, color=color); a.set_title(name); a.set_xlabel("z")
    a.axvline(z, ls="--", color="gray"); a.scatter([z], [f(z)], color="k", zorder=3)
plt.tight_layout(); plt.show()

**Qué deberías ver:** escalón = 1 (aprueba), sigmoide ≈ **0.943** (94 % de probabilidad de pago), ReLU = 2.81. En las gráficas, el punto negro marca nuestro socio.

### Tarea 2 · De puntaje a decisión
Con el `z_b` de la Tarea 1:
1. Calcula la probabilidad de pago del socio B con la sigmoide → `p_b`.
2. Decide con umbral 0.5: si `p_b >= 0.5` escribe `"aprueba"`, si no `"rechaza"` → `decision_b`.

In [ ]:
p_b = None           # TODO: sigmoid of z_b
decision_b = None    # TODO: "aprueba" or "rechaza"
print("Probabilidad de pago del socio B:", p_b, "→", decision_b)

In [ ]:
revisar_t2(p_b, decision_b)

---
## 3 · El perceptrón aprende de UN error
**Idea:** el perceptrón (Rosenblatt, 1958) aprende corrigiéndose **solo cuando se equivoca**. Usamos etiquetas $y = +1$ (aprobar) y $y = -1$ (rechazar):

$$\mathbf{w} \leftarrow \mathbf{w} + \eta \cdot y \cdot \mathbf{x} \qquad b \leftarrow b + \eta \cdot y$$

En palabras: **empuja los pesos hacia la respuesta correcta**, en la dirección del dato que clasificó mal.

### Ejemplo resuelto: debía aprobar y rechazó ($y = +1$)

In [ ]:
w = np.array([0.2, -0.5]); b = 0.0; eta = 1.0     # current weights (a "new" committee member)
x_i = np.array([1.0, 1.0]); y_i = +1              # a member who SHOULD be approved

z_before = w @ x_i + b
print(f"Antes:   z = {z_before:+.2f} -> predice {'aprobar' if z_before >= 0 else 'rechazar'} (correcto: aprobar)")

if y_i * z_before <= 0:          # mistake -> correct the weights
    w = w + eta * y_i * x_i
    b = b + eta * y_i
print(f"Pesos corregidos: w = {w}, b = {b}")

z_after = w @ x_i + b
print(f"Después: z = {z_after:+.2f} -> predice {'aprobar' if z_after >= 0 else 'rechazar'}")

**Qué deberías ver:** antes z = **−0.30** (se equivoca y rechaza); después de corregir w = (1.2, 0.5), b = 1 y z = **+2.70** (ahora aprueba).

### Tarea 3 · Corrige un error al revés ($y = -1$)
Ahora el comité tiene `w = (1.2, 0.5)` y `b = 1`. Llega un socio `x = (1.0, 0.5)` que **no pagó** ($y = -1$), pero la neurona lo aprueba (z = 2.45).
1. Aplica **una** corrección con η = 1 → `w_nuevo`, `b_nuevo`.
2. Con los pesos nuevos, ¿todavía se equivoca? Escribe `True` o `False` en `todavia_se_equivoca`.

<details><summary>Pista</summary>

Como $y = -1$, la regla **resta** el dato: $\mathbf{w} + 1\cdot(-1)\cdot\mathbf{x} = \mathbf{w} - \mathbf{x}$.
</details>

In [ ]:
w = np.array([1.2, 0.5]); b = 1.0; eta = 1.0
x_i = np.array([1.0, 0.5]); y_i = -1

w_nuevo = None                # TODO: w + eta * y_i * x_i
b_nuevo = None                # TODO: b + eta * y_i
todavia_se_equivoca = None    # TODO: True or False (compute z with the new weights)

In [ ]:
revisar_t3(w_nuevo, b_nuevo, todavia_se_equivoca)

---
## 4 · Entrenar un perceptrón completo
**Idea:** repetir la regla sobre todos los ejemplos, una y otra vez (cada vuelta completa es una **época**), hasta que no haya errores.

### Ejemplo resuelto
Generamos dos variables de socios y una regla "verdadera" que separa aprobados de rechazados con un margen.

In [ ]:
n = 120
X = np.random.uniform(-1, 1, size=(n, 2))
margin = X @ np.array([1.0, 1.6]) - 0.1
keep = np.abs(margin) > 0.25                       # leave a gap between the classes
X, y = X[keep], np.where(margin[keep] > 0, 1, -1)

def train_perceptron(X, y, eta=1.0, epochs=50):
    w, b, history = np.zeros(X.shape[1]), 0.0, []
    for epoch in range(epochs):
        mistakes = 0
        for xi, yi in zip(X, y):
            if yi * (w @ xi + b) <= 0:             # wrong (or undecided) -> correct
                w, b = w + eta * yi * xi, b + eta * yi
                mistakes += 1
        history.append(mistakes)
        if mistakes == 0:                          # a full pass without errors: done
            break
    return w, b, history

w_p, b_p, hist = train_perceptron(X, y)
print("Errores por época:", hist)

xs = np.linspace(-1, 1, 50)
plt.figure(figsize=(5.5, 5))
plt.scatter(*X[y == 1].T, c="#1F55E0", label="aprobar (+1)", edgecolor="k", lw=.3)
plt.scatter(*X[y == -1].T, c="#D97706", label="rechazar (−1)", edgecolor="k", lw=.3)
plt.plot(xs, -(w_p[0] * xs + b_p) / w_p[1], "k-", lw=2.5, label="línea aprendida")
plt.ylim(-1.1, 1.1); plt.legend(loc="lower left"); plt.title("El perceptrón encontró una línea"); plt.show()

**Qué deberías ver:** errores por época **[2, 0]** y una línea negra que separa azules de naranjas.
Esto es lo que garantiza el teorema de Novikoff (1962): *si existe una línea que separa los datos con margen, el perceptrón la encuentra en un número finito de correcciones.*

### Tarea 4 · Aprende la compuerta AND
AND es "los dos a la vez": vale $+1$ solo si ambos interruptores están en 1.
1. Escribe los 4 casos en `X_and` y sus respuestas en `y_and` (+1 o −1).
2. Entrena con la función `train_perceptron` de arriba → `w_and, b_and, hist_and`.

| $x_1$ | $x_2$ | AND |
|---|---|---|
| 0 | 0 | −1 |
| 0 | 1 | −1 |
| 1 | 0 | −1 |
| 1 | 1 | +1 |

In [ ]:
X_and = None    # TODO: np.array with the 4 rows [x1, x2]
y_and = None    # TODO: np.array with -1 / +1
w_and, b_and, hist_and = None, None, None    # TODO: train_perceptron(X_and, y_and)

In [ ]:
revisar_t4(w_and, b_and, X_and, y_and)

---
## 5 · El límite: XOR, la luz de la escalera
**Idea:** dos interruptores, uno abajo y otro arriba de la escalera. La luz se enciende si están en posiciones **distintas**. Eso es XOR.

| $x_1$ | $x_2$ | XOR |
|---|---|---|
| 0 | 0 | 0 (apagada) |
| 0 | 1 | 1 (encendida) |
| 1 | 0 | 1 (encendida) |
| 1 | 1 | 0 (apagada) |

### Ejemplo resuelto: el perceptrón intenta aprender XOR

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y_xor = np.array([-1, 1, 1, -1])
_, _, hist_xor = train_perceptron(X_xor, y_xor, epochs=20)
print("Errores por época:", hist_xor)
plt.figure(figsize=(6, 3)); plt.plot(range(1, len(hist_xor) + 1), hist_xor, "o-", color="#D97706")
plt.ylim(0, 4.2); plt.xlabel("época"); plt.ylabel("errores"); plt.title("XOR: nunca llega a 0 errores"); plt.show()

**Qué deberías ver:** **4 errores en todas las épocas**: nunca mejora.

### Tarea 5 · ¿Por qué falla?
Elige la mejor explicación y escribe su letra en `respuesta_t5`:

- **a)** Le faltan épocas: con 1 000 épocas lo lograría.
- **b)** No existe una línea recta que separe las luces encendidas de las apagadas.
- **c)** La tasa de aprendizaje η es demasiado grande.
- **d)** Los datos tienen ruido.

En el negocio esto aparece cuando el efecto de una variable **depende de otra** (por ejemplo, el garante ayuda solo si la cuota es baja). Mañana lo resolvemos con **más de una neurona**.

In [ ]:
respuesta_t5 = None    # TODO: "a", "b", "c" or "d"

In [ ]:
revisar_t5(respuesta_t5)

---
## 6 · La pérdida: ¿qué tan mal lo hicimos?
**Idea:** para aprender necesitamos **un solo número** que mida el error. Es como una nota al revés: 0 es perfecto.

**Error cuadrático medio (MSE)**, para predecir números (por ejemplo, precios):
$$\text{MSE} = \frac{1}{N}\sum_{i=1}^{N} (y_i - \hat y_i)^2$$

**Entropía cruzada binaria (BCE)**, para predecir sí/no con una probabilidad $\hat y$:
$$\text{BCE} = -\big[\,y\log\hat y + (1-y)\log(1-\hat y)\,\big]$$
Si pagó ($y=1$) queda $-\log\hat y$; si no pagó ($y=0$) queda $-\log(1-\hat y)$. Castiga muchísimo estar **seguro y equivocado**.

### Ejemplo resuelto

In [ ]:
# MSE: predicted vs real house prices in Quito (thousands of USD)
real = np.array([120, 95, 150, 80])
pred = np.array([110, 100, 170, 82])
errors = real - pred
print("Errores:", errors, "| al cuadrado:", errors**2, "| MSE =", np.mean(errors**2))

# BCE: the real answer is y = 1 ("sí pagó"). How much does each prediction cost?
for p in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"El modelo dijo {p:4.2f} -> pérdida = -log({p}) = {-np.log(p):.2f}")

**Qué deberías ver:** MSE = **132.25** (la casa 3, con error 20, aporta 400 de los 529). En la BCE: 0.99 → 0.01, 0.5 → 0.69 y 0.01 → **4.61**. Estar muy seguro y equivocado cuesta caro, como pronosticar "99 % sin lluvia" el día de un aguacero en Quito.

### Tarea 6 · Programa MSE y BCE
1. `mse(real, pred)`: el promedio de los errores al cuadrado.
2. `bce(y, p)`: la fórmula completa de arriba, para **un** dato con $y \in \{0, 1\}$.

<details><summary>Pista</summary>

`np.mean(...)`, `** 2` y `np.log(...)` son todo lo que necesitas.
</details>

In [ ]:
def mse(real, pred):
    return None    # TODO

def bce(y, p):
    return None    # TODO

In [ ]:
revisar_t6(mse, bce)

---
## 7 · Descenso de gradiente: bajar la montaña
**Idea:** como bajar el Cotopaxi con neblina: no ves el valle, pero con el pie sientes hacia dónde baja el terreno (la **pendiente**) y das un paso hacia allá.

$$w_{\text{nuevo}} = w - \eta \cdot \text{pendiente}$$

### Ejemplo resuelto
Pérdida $L(w) = (w - 3)^2$, con su mínimo en $w = 3$. Su pendiente (derivada) es $2(w-3)$. Empezamos en $w = 0$ con $\eta = 0.1$.

In [ ]:
L = lambda w: (w - 3) ** 2
slope = lambda w: 2 * (w - 3)

w, eta, path = 0.0, 0.1, [0.0]
print(" paso |    w    | pendiente | pérdida")
for step_i in range(10):
    print(f"  {step_i:2d}  | {w:7.3f} | {slope(w):9.3f} | {L(w):7.3f}")
    w = w - eta * slope(w)
    path.append(w)

ws = np.linspace(-0.5, 5, 200)
plt.figure(figsize=(6.5, 3.6)); plt.plot(ws, L(ws), lw=2.5, color="#1F55E0")
plt.plot(path, [L(p) for p in path], "o-", color="#D97706", ms=6, label="pasos")
plt.xlabel("w"); plt.ylabel("pérdida"); plt.legend(); plt.title("Cada paso baja un poco"); plt.show()

**Qué deberías ver:** w = 0 → 0.6 → 1.08 → 1.464 → 1.771 … acercándose a **3**, y la pérdida bajando de 9 hacia 0. Los pasos se acortan solos porque la pendiente se suaviza cerca del fondo.

### Tarea 7 · Baja otra montaña
Ahora la pérdida es $L(w) = (w + 2)^2$.
1. ¿En qué valor de $w$ está el mínimo? → `minimo`.
2. Su pendiente es $2(w+2)$. Empieza en $w = 5$ y da **50 pasos** con $\eta = 0.1$. Guarda el último valor en `w_final`.

In [ ]:
minimo = None     # TODO: where is the bottom?
w = 5.0
eta = 0.1
# TODO: write a loop of 50 steps: w = w - eta * slope, with slope = 2 * (w + 2)

w_final = None    # TODO: the value of w after the loop

In [ ]:
revisar_t7(minimo, w_final)

---
## 8 · El tamaño del paso importa
**Idea:** si el paso η es muy corto, tardas una eternidad; si es muy largo, te pasas del fondo y rebotas cada vez más lejos.

### Ejemplo resuelto: misma montaña, tres pasos distintos

In [ ]:
ws_wide = np.linspace(-3, 9, 300)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for a, eta, title in zip(ax, [0.02, 0.4, 1.02], ["muy pequeño: lento", "bueno", "muy grande: diverge"]):
    w, path = 0.0, [0.0]
    for _ in range(12):
        w = w - eta * slope(w); path.append(w)
    a.plot(ws_wide, L(ws_wide), lw=2.5, color="#1F55E0")
    a.plot(path, [L(p) for p in path], "o-", color="#D97706", ms=5)
    a.set_ylim(-1, 40); a.set_title(f"η = {eta}: {title}"); a.set_xlabel("w")
plt.tight_layout(); plt.show()

def descenso(eta, pasos=30, w=0.0):
    # gradient descent on L(w) = (w - 3)^2; returns the final w
    for _ in range(pasos):
        w = w - eta * slope(w)
    return w

for eta in [0.02, 0.4, 1.02]:
    print(f"η = {eta:4}: después de 30 pasos w = {descenso(eta):10.3f}  (el mínimo está en 3)")

**Qué deberías ver:** con η = 0.02 apenas avanza; con η = 0.4 llega rápido; con η = 1.02 **rebota de lado a lado, cada vez más lejos** (diverge).

### Tarea 8 · ¿Cuáles convergen?
Usa `descenso(eta)` para probar cada valor y escribe `True` si **se acerca** al mínimo (aunque sea lento o rebotando) o `False` si **se aleja**.

In [ ]:
# try: print(descenso(0.9))
converge = {0.05: None, 0.5: None, 0.9: None, 1.1: None}    # TODO: True / False

In [ ]:
revisar_t8(converge)

---
## 🏁 Mini-reto 1 · Tu propio comité de crédito
Seis socios de la cooperativa, con dos datos escalados: **ingreso** y **atrasos previos**. Tres pagaron (azul) y tres no (naranja).

| Socio | Ingreso | Atrasos | ¿Pagó? |
|---|---|---|---|
| Ana | 0.9 | −0.8 | sí |
| Byron | 0.4 | −0.2 | sí |
| Carla | 0.8 | 0.3 | sí |
| Diego | −0.5 | 0.6 | no |
| Elena | 0.1 | 0.5 | no |
| Fabián | −0.3 | −0.1 | no |

**Tu misión:** elige a mano los pesos `w = [w_ingreso, w_atrasos]` y el sesgo `b` para que tu neurona apruebe a los que pagaron y rechace a los que no. La celda dibuja tu frontera y cuenta aciertos.

**Para pensar:**
1. ¿Qué signo debería tener el peso de los atrasos? ¿Por qué?
2. Carla está cerca de la frontera: ¿qué pasa con ella si haces al comité más exigente (bajas $b$)?
3. **Bonus:** entrena con `train_perceptron(SOCIOS, PAGO)` y compara sus pesos con los tuyos.

In [ ]:
w_comite = None    # TODO: e.g. [1.0, -1.0]
b_comite = None    # TODO: e.g. 0.0
revisar_mini_reto_1(w_comite, b_comite)

---
### Para llevar
1. Una neurona **multiplica, suma y decide**: $z = \mathbf{w}\cdot\mathbf{x} + b$ y luego una activación.
2. El perceptrón aprende **corrigiendo sus errores**, pero una sola neurona solo traza **una línea**: no resuelve XOR.
3. La **pérdida** mide el error; el **descenso de gradiente** la reduce paso a paso; el tamaño del paso η importa.

**Mañana:** juntamos neuronas en capas, vemos cómo aprende una red completa (retropropagación) y pasamos a Keras con el **NB02**.